# Denver CPI Trends Analysis

Consumer Price Index analysis for Denver/Boulder/Greeley metro area (BLS extract).

- **Frequency in this file**: Denver metro rows are **Annual** and **Semi-annual** only (H1 / H2). There is **no monthly** Denver CPI in this dataset; semi-annual is the highest frequency available here.
- **Headline vs components**: CPI-U all items plus major **product categories** (food, housing, energy, etc.).
- **Inflation**: Year-over-year (YoY) from annual averages vs YoY from matching semi-annual periods (same half, prior year).

## Setup & Data Loading

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import display, HTML

# Prism palette (same pattern as wine-reviews notebook)
PRISM = list(px.colors.qualitative.Prism)
px.defaults.color_discrete_sequence = PRISM
px.defaults.color_continuous_scale = [[i / (len(PRISM) - 1), c] for i, c in enumerate(PRISM)]

# Kaggle (and some hosted notebooks) sanitize display(HTML(...)) and block Plotly's CDN <script>.
# Use Plotly's "iframe" renderer + fig.show() on Kaggle — https://www.kaggle.com/code/stpeteishii/solution-to-a-plotly-graph-cannot-be-displayed
_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
if _IS_KAGGLE:
    pio.renderers.default = "iframe"


def show_plotly(fig):
    # Optional: PLOTLY_FORCE_HTML=1 to keep CDN HTML (e.g. some local viewers).
    if os.environ.get("PLOTLY_FORCE_HTML", "").lower() in ("1", "true", "yes"):
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    elif _IS_KAGGLE:
        fig.show()
    else:
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


# Resolve data path: Kaggle first, then local
KAGGLE_PATH = Path("/kaggle/input/datasets/organizations/bls/denver-cpi/Consumer_Price_Index_in_Denver.csv")
LOCAL_PATH = Path("data/Consumer_Price_Index_in_Denver.csv")

if KAGGLE_PATH.exists():
    csv_path = KAGGLE_PATH
    print(f"Using Kaggle path: {csv_path}")
else:
    csv_path = LOCAL_PATH
    if not csv_path.exists():
        csv_path = Path.cwd() / "data" / "Consumer_Price_Index_in_Denver.csv"
    print(f"Using local path: {csv_path}")

df_raw = pd.read_csv(csv_path)

Using local path: data/Consumer_Price_Index_in_Denver.csv


In [2]:
# Colorado / Denver metro — all product types for category charts; `denver` = headline CPI-U all items only
co = df_raw[df_raw["dataRegion"] == "CO"].copy()
TYPE_LABELS = {
    1: "All items (CPI-U)",
    51: "Food and beverages",
    52: "Housing",
    53: "Fuels and utilities",
    54: "Apparel",
    55: "Transportation",
    56: "Medical care",
    57: "Recreation",
    58: "Education and communication",
    59: "Other goods and services",
}
co["product"] = co["type"].map(TYPE_LABELS)

denver = co[co["type"] == 1].copy()
denver.head(10)

,stateFips,area,areaType,period,periodYear,periodType,periodTypeDescription,cpi,title,type,source,cpiSourceDescription,percentChangeYear,percentChangeMonth,dataRegion,areaName,areaDescription,product
8,8,2,24,6,1996,7,Semi-Annual,152.0,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",3.5,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
17,8,2,24,12,1996,7,Semi-Annual,154.2,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",3.5,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
26,8,2,24,0,1997,1,Annual,158.1,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",3.3,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
35,8,2,24,6,1997,7,Semi-Annual,157.1,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",3.4,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
52,8,2,24,12,1997,7,Semi-Annual,159.1,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",3.2,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
61,8,2,24,0,1998,1,Annual,161.9,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",2.4,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
70,8,2,24,6,1998,7,Semi-Annual,160.5,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",2.2,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
79,8,2,24,12,1998,7,Semi-Annual,163.3,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",2.6,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
88,8,2,24,0,1999,1,Annual,166.6,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",2.9,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)
313,8,2,24,6,1999,7,Semi-Annual,165.1,"CPI-U all items 1982-84=100, not seasonally ad...",1,1,"US DOL, Bureau of Labor Statistics",2.9,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U)


In [3]:
# Data overview: period types available for Denver
denver.groupby(["periodType", "periodTypeDescription"]).size()

periodType  periodTypeDescription
1           Annual                   18
7           Semi-Annual              37
dtype: int64

## Annual Trends

In [4]:
# Annual data: periodType 1 = Annual
annual = denver[denver["periodType"] == 1].copy()
annual = annual.groupby("periodYear", as_index=False)["cpi"].mean()
annual = annual.sort_values("periodYear").reset_index(drop=True)

# YoY percent change
annual["pct_change_yoy"] = annual["cpi"].pct_change() * 100
annual.head(15)

,periodYear,cpi,pct_change_yoy
0,1995,147.9,NaN
1,1996,153.1,3.515889
2,1997,158.1,3.265839
3,1998,161.9,2.403542
4,1999,166.6,2.903027
5,2000,173.2,3.961585
6,2001,181.3,4.676674
7,2002,184.8,1.930502
8,2003,186.8,1.082251
9,2004,187.0,0.107066


In [5]:
fig = px.line(
    annual,
    x="periodYear",
    y="cpi",
    markers=True,
    title="Denver CPI — annual index (CPI-U all items, 1982-84=100)",
    labels={"periodYear": "Year", "cpi": "CPI"},
)
fig.update_traces(line_color=PRISM[0], marker=dict(size=7))
fig.update_layout(template="plotly_white", height=450, hovermode="x unified")
show_plotly(fig)

In [6]:
# Year-over-year inflation (annual averages)
yoy = annual.dropna(subset=["pct_change_yoy"]).copy()
yoy["color"] = yoy["pct_change_yoy"].apply(lambda v: PRISM[2] if v >= 0 else PRISM[0])
fig = go.Figure(
    go.Bar(
        x=yoy["periodYear"],
        y=yoy["pct_change_yoy"],
        marker_color=yoy["color"],
        name="Annual YoY %",
    )
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Denver CPI — annual year-over-year % change (inflation)",
    xaxis_title="Year",
    yaxis_title="YoY %",
    template="plotly_white",
    height=450,
    showlegend=False,
)
show_plotly(fig)

## Semi-Annual (Sub-Annual) Trends

In [7]:
# Semi-annual data: periodType 7, period 6 = H1, period 12 = H2
semi = denver[denver["periodType"] == 7].copy()
semi["half"] = semi["period"].map({6: "H1 (Jan-Jun)", 12: "H2 (Jul-Dec)"})
semi["period_label"] = semi["periodYear"].astype(str) + " " + semi["half"]
semi = semi.sort_values(["periodYear", "period"]).reset_index(drop=True)

# Quarter-like label for display (H1/H2)
semi["year_half"] = semi["periodYear"].astype(str) + "-" + semi["period"].astype(str)
semi.head(10)

,stateFips,area,areaType,period,periodYear,periodType,periodTypeDescription,cpi,title,type,...,cpiSourceDescription,percentChangeYear,percentChangeMonth,dataRegion,areaName,areaDescription,product,half,period_label,year_half
0,8,2,24,6,1995,7,Semi-Annual,146.9,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",4.9,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H1 (Jan-Jun),1995 H1 (Jan-Jun),1995-6
1,8,2,24,12,1995,7,Semi-Annual,149.0,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",3.8,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H2 (Jul-Dec),1995 H2 (Jul-Dec),1995-12
2,8,2,24,6,1996,7,Semi-Annual,152.0,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",3.5,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H1 (Jan-Jun),1996 H1 (Jan-Jun),1996-6
3,8,2,24,12,1996,7,Semi-Annual,154.2,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",3.5,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H2 (Jul-Dec),1996 H2 (Jul-Dec),1996-12
4,8,2,24,6,1997,7,Semi-Annual,157.1,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",3.4,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H1 (Jan-Jun),1997 H1 (Jan-Jun),1997-6
5,8,2,24,12,1997,7,Semi-Annual,159.1,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",3.2,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H2 (Jul-Dec),1997 H2 (Jul-Dec),1997-12
6,8,2,24,6,1998,7,Semi-Annual,160.5,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",2.2,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H1 (Jan-Jun),1998 H1 (Jan-Jun),1998-6
7,8,2,24,12,1998,7,Semi-Annual,163.3,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",2.6,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H2 (Jul-Dec),1998 H2 (Jul-Dec),1998-12
8,8,2,24,6,1999,7,Semi-Annual,165.1,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",2.9,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H1 (Jan-Jun),1999 H1 (Jan-Jun),1999-6
9,8,2,24,12,1999,7,Semi-Annual,168.2,"CPI-U all items 1982-84=100, not seasonally ad...",1,...,"US DOL, Bureau of Labor Statistics",3.0,0.0,CO,Denver/Boulder/Greeley,Denver CPI region,All items (CPI-U),H2 (Jul-Dec),1999 H2 (Jul-Dec),1999-12


In [8]:
fig = px.line(
    semi,
    x="period_label",
    y="cpi",
    markers=True,
    title="Denver CPI — semi-annual index (H1 & H2, headline)",
    labels={"period_label": "Period", "cpi": "CPI"},
)
fig.update_traces(line_color=PRISM[4], marker=dict(size=6))
fig.update_layout(template="plotly_white", height=480, xaxis_tickangle=-45, hovermode="x unified")
show_plotly(fig)

In [9]:
# Combined: annual average vs semi-annual observations (same index concept, different sampling)
semi_plot = semi.copy()
semi_plot["x_pos"] = semi_plot["periodYear"] + (semi_plot["period"] / 12 - 0.5)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=annual["periodYear"],
        y=annual["cpi"],
        mode="lines+markers",
        name="Annual",
        line=dict(color=PRISM[0], width=2),
        marker=dict(size=8),
    )
)
fig.add_trace(
    go.Scatter(
        x=semi_plot["x_pos"],
        y=semi_plot["cpi"],
        mode="markers",
        name="Semi-annual (H1/H2)",
        marker=dict(color=PRISM[4], symbol="diamond", size=9),
        text=semi_plot["period_label"],
        hovertemplate="%{text}<br>CPI=%{y:.1f}<extra></extra>",
    )
)
fig.update_layout(
    title="Denver CPI — annual vs semi-annual (headline index)",
    xaxis_title="Year (semi-annual points placed mid-half)",
    yaxis_title="CPI (1982-84=100)",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
show_plotly(fig)

## Inflation: annual YoY vs semi-annual YoY

**Annual YoY** uses consecutive *calendar-year* averages. **Semi-annual YoY** compares each H1 (or H2) index to the *same half* one year earlier—closer to what people think of as “twice-a-year” inflation readings. This file does not include **monthly** Denver CPI; semi-annual is the finest grain here.

In [10]:
# Semi-annual YoY: same period code (6 or 12) vs prior year — headline only (uses `semi` from above)
semi_base = semi[["periodYear", "period", "cpi", "period_label"]].copy()
prev_cpi = semi_base[["periodYear", "period", "cpi"]].rename(columns={"cpi": "cpi_prev"})
prev_cpi["periodYear"] = prev_cpi["periodYear"] + 1
semi_yoy = semi_base.merge(prev_cpi, on=["periodYear", "period"], how="left")
semi_yoy["semi_yoy_pct"] = (semi_yoy["cpi"] / semi_yoy["cpi_prev"] - 1) * 100
semi_yoy = semi_yoy.dropna(subset=["semi_yoy_pct"])

# Align on a simple date axis (mid-half)
month_map = {6: "04-15", 12: "10-15"}
semi_yoy["date"] = pd.to_datetime(
    semi_yoy["periodYear"].astype(str) + "-" + semi_yoy["period"].map(month_map)
)

annual_yoy = annual.dropna(subset=["pct_change_yoy"]).copy()
annual_yoy["date"] = pd.to_datetime(annual_yoy["periodYear"].astype(str) + "-07-01")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=annual_yoy["date"],
        y=annual_yoy["pct_change_yoy"],
        mode="lines+markers",
        name="Annual YoY %",
        line=dict(color=PRISM[0], width=2),
        marker=dict(size=7),
    )
)
fig.add_trace(
    go.Scatter(
        x=semi_yoy["date"],
        y=semi_yoy["semi_yoy_pct"],
        mode="lines+markers",
        name="Semi-annual YoY % (vs same H1/H2, prior year)",
        line=dict(color=PRISM[4], width=2),
        marker=dict(size=6),
        text=semi_yoy["period_label"],
        hovertemplate="%{text}<br>YoY=%{y:.2f}%<extra></extra>",
    )
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Headline inflation: annual vs semi-annual YoY (Denver)",
    xaxis_title="Date (positioned for readability)",
    yaxis_title="YoY %",
    template="plotly_white",
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    hovermode="x unified",
)
show_plotly(fig)

## CPI by product type (annual series)

Major BLS categories for Denver (annual observations). **Semi-annual** series exist for each category as well; headline semi-annual paths are plotted above.

In [11]:
ann_cat = co[co["periodType"] == 1].sort_values(["product", "periodYear"])
fig = px.line(
    ann_cat,
    x="periodYear",
    y="cpi",
    color="product",
    title="Denver CPI by category — annual index (1982-84=100)",
    labels={"periodYear": "Year", "cpi": "CPI", "product": "Category"},
)
fig.update_layout(template="plotly_white", height=520, legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02))
show_plotly(fig)

In [12]:
# Latest calendar-year YoY by category (annual averages)
ann_cat = co[co["periodType"] == 1].sort_values(["type", "periodYear"])
ann_cat["yoy_pct"] = ann_cat.groupby("type")["cpi"].pct_change() * 100
latest = ann_cat.groupby("type", as_index=False).last()
latest = latest.dropna(subset=["yoy_pct"]).sort_values("yoy_pct")
latest["bar_color"] = latest["yoy_pct"].apply(lambda v: PRISM[2] if v >= 0 else PRISM[0])

fig = go.Figure(
    go.Bar(
        y=latest["product"],
        x=latest["yoy_pct"],
        orientation="h",
        marker_color=latest["bar_color"],
        text=latest["yoy_pct"].round(2).astype(str) + "%",
        textposition="outside",
    )
)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="YoY inflation by category — latest annual vs prior year (latest calendar year can differ by category)",
    xaxis_title="YoY %",
    yaxis_title=None,
    template="plotly_white",
    height=420,
    margin=dict(l=20, r=80),
)
show_plotly(fig)

latest[["product", "periodYear", "cpi", "yoy_pct"]].round(2)

,product,periodYear,cpi,yoy_pct
9,Other goods and services,2011,333.70,0.18
8,Education and communication,2011,118.70,1.02
6,Medical care,2011,452.30,1.32
0,All items (CPI-U),2012,224.57,1.94
2,Housing,2011,198.00,2.22
4,Apparel,2011,101.50,2.73
1,Food and beverages,2011,208.90,3.98
7,Recreation,2011,143.90,4.50
3,Fuels and utilities,2011,205.90,7.63
5,Transportation,2011,260.30,10.11


## YoY inflation by year and category

Annual **calendar-year** YoY for every published Denver category. The line chart tracks which sectors drive each year’s inflation; the heatmap highlights high/low regimes. **Analysis** tables summarize typical volatility (std of YoY), extremes, and how each category’s YoY co-moves with headline CPI.

In [13]:
# Panel: YoY from consecutive annual index levels (same definition as earlier cells)
ann_yoy = co[co["periodType"] == 1].sort_values(["type", "periodYear"]).copy()
ann_yoy["yoy_pct"] = ann_yoy.groupby("type")["cpi"].pct_change() * 100
yoy_long = ann_yoy.dropna(subset=["yoy_pct"])

fig = px.line(
    yoy_long,
    x="periodYear",
    y="yoy_pct",
    color="product",
    markers=True,
    title="Denver — annual YoY % by category (calendar-year averages)",
    labels={"periodYear": "Year", "yoy_pct": "YoY %", "product": "Category"},
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_white", height=560, hovermode="x unified", legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02))
show_plotly(fig)

heat = yoy_long.pivot_table(index="product", columns="periodYear", values="yoy_pct")
diverging = [[0, PRISM[10]], [0.5, "#f0f0f0"], [1, PRISM[2]]]
fig_h = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale=diverging,
    title="YoY % heatmap: category × year",
    labels=dict(x="Year", y="Category", color="YoY %"),
)
fig_h.update_layout(height=480)
show_plotly(fig_h)

# --- Analysis ---
by_cat = yoy_long.groupby("product")["yoy_pct"]
volatility = by_cat.agg(mean_yoy="mean", std_yoy="std", min_yoy="min", max_yoy="max", n_years="count").sort_values("std_yoy", ascending=False)
display(volatility.round(2))

wide = yoy_long.pivot(index="periodYear", columns="product", values="yoy_pct").dropna(how="all")
headline = "All items (CPI-U)"
if headline in wide.columns:
    with_headline = wide.corrwith(wide[headline]).drop(labels=[headline], errors="ignore").sort_values(ascending=False)
    display(with_headline.round(3).to_frame("corr_with_headline_yoy"))

print(
    "Most volatile YoY (highest std):",
    volatility.index[0],
    f"({volatility['std_yoy'].iloc[0]:.2f} pp).",
    "Calmest:",
    volatility.index[-1],
    f"({volatility['std_yoy'].iloc[-1]:.2f} pp).",
)

,mean_yoy,std_yoy,min_yoy,max_yoy,n_years
product,,,,,
Apparel,3.04,8.51,-4.08,19.26,6
Fuels and utilities,1.07,7.55,-8.98,10.53,6
Transportation,4.38,6.07,-6.82,10.11,6
Medical care,4.30,3.15,0.79,7.45,6
Food and beverages,2.24,2.04,-0.20,4.73,7
All items (CPI-U),2.50,1.42,-0.67,4.68,17
Housing,1.56,1.42,0.00,3.60,6
Education and communication,2.25,1.41,0.69,4.45,6
Other goods and services,2.59,1.27,0.18,3.61,6


,corr_with_headline_yoy
product,
Transportation,0.829
Housing,0.786
Fuels and utilities,0.775
Food and beverages,0.663
Apparel,0.445
Medical care,0.434
Recreation,0.002
Other goods and services,-0.280
Education and communication,-0.363


Most volatile YoY (highest std): Apparel (8.51 pp). Calmest: Recreation (0.90 pp).


## Radar: compare category profiles

This extract has **one BLS index per category** (not a basket of sub-products). The radar uses four **relative** axes (each min–max scaled to 0–100 across categories so shapes are comparable):

1. **Latest CPI** — relative price level (index is not comparable in levels across categories, but rank-style normalization still shows which series sits high vs low in *this* cross-section).
2. **Mean YoY** — average calendar-year inflation for that category over its sample.
3. **YoY volatility** — standard deviation of annual YoY (inflation “choppiness”).
4. **Years in sample** — count of annual rows (categories start in different years here; more years ⇒ richer history).

Use the legend to toggle categories. *“Number of products”* is not in the file; **years in sample** is the available “how much data / how established the series is” proxy.

In [14]:
def _minmax_scale(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if np.isclose(hi, lo):
        return pd.Series(50.0, index=s.index)
    return (s - lo) / (hi - lo) * 100.0


radar_rows = []
for typ, g in ann_yoy.groupby("type"):
    g = g.sort_values("periodYear")
    yy = g["yoy_pct"].dropna()
    radar_rows.append(
        {
            "product": g["product"].iloc[0],
            "cpi_latest": float(g["cpi"].iloc[-1]),
            "mean_yoy": float(yy.mean()) if len(yy) else np.nan,
            "std_yoy": float(yy.std(ddof=0)) if len(yy) > 1 else 0.0,
            "n_years": int(len(g)),
        }
    )
radar_df = pd.DataFrame(radar_rows).dropna(subset=["mean_yoy"])

radar_df["r_cpi"] = _minmax_scale(radar_df["cpi_latest"])
radar_df["r_mean"] = _minmax_scale(radar_df["mean_yoy"])
radar_df["r_std"] = _minmax_scale(radar_df["std_yoy"])
radar_df["r_n"] = _minmax_scale(radar_df["n_years"].astype(float))

theta = [
    "Latest CPI<br>(scaled)",
    "Mean YoY<br>(scaled)",
    "YoY volatility<br>(scaled)",
    "Years in<br>sample",
]
theta_closed = theta + [theta[0]]

fig_r = go.Figure()
for _, row in radar_df.iterrows():
    rvals = [row["r_cpi"], row["r_mean"], row["r_std"], row["r_n"], row["r_cpi"]]
    fig_r.add_trace(
        go.Scatterpolar(
            r=rvals,
            theta=theta_closed,
            mode="lines",
            name=row["product"],
            fill="toself",
            opacity=0.08,
            line=dict(width=1.5),
        )
    )

fig_r.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100], tickvals=[0, 25, 50, 75, 100])),
    title="Category comparison radar (0–100 = min–max within Denver categories)",
    height=720,
    template="plotly_white",
    legend=dict(orientation="v", yanchor="middle", y=0.5, x=1.02, xanchor="left"),
)
show_plotly(fig_r)

radar_df[["product", "cpi_latest", "mean_yoy", "std_yoy", "n_years"]].round(2)

,product,cpi_latest,mean_yoy,std_yoy,n_years
0,All items (CPI-U),224.57,2.50,1.38,18
1,Food and beverages,208.90,2.24,1.88,8
2,Housing,198.00,1.56,1.29,7
3,Fuels and utilities,205.90,1.07,6.89,7
4,Apparel,101.50,3.04,7.77,7
5,Transportation,260.30,4.38,5.54,7
6,Medical care,452.30,4.30,2.88,7
7,Recreation,143.90,3.31,0.82,7
8,Education and communication,118.70,2.25,1.29,7
9,Other goods and services,333.70,2.59,1.16,7


## Per-category deep dive (spread & behavior)

The CSV publishes **one index per category** (no item-level children). Within each category we still observe **behavior** through: (1) annual CPI vs YoY, (2) the **distribution** of YoY draws, (3) **semi-annual H1/H2** path, and (4) **intra-year swing** \(|H2-H1|/H1\) when both halves exist.

Adjust `DEEP_DIVE_PRODUCTS` to limit output if the notebook gets too long.

In [15]:
from IPython.display import Markdown

# Set to a list of product names to restrict plots; None = all categories
DEEP_DIVE_PRODUCTS = None  # e.g. ["Housing", "Transportation", "All items (CPI-U)"]

semi_all = co[co["periodType"] == 7].copy()
semi_all["half_label"] = semi_all["period"].map({6: "H1", 12: "H2"})
semi_all["period_label"] = semi_all["periodYear"].astype(str) + " " + semi_all["half_label"]
semi_all = semi_all.sort_values(["product", "periodYear", "period"])

pv_spread = semi_all.pivot_table(index=["product", "periodYear"], columns="half_label", values="cpi", aggfunc="first")
if "H1" in pv_spread.columns and "H2" in pv_spread.columns:
    pv_spread = pv_spread.assign(
        intra_year_swing_pct=(pv_spread["H2"] - pv_spread["H1"]).abs() / pv_spread["H1"] * 100.0
    )
    swing_tbl = (
        pv_spread.groupby("product")["intra_year_swing_pct"]
        .agg(mean_swing_pct="mean", std_swing_pct="std", max_swing_pct="max")
        .round(2)
        .sort_values("mean_swing_pct", ascending=False)
    )
    display(Markdown("**Summary: intra-year swing** — mean / std / max of |H2−H1|/H1 (semi-annual, within calendar year)"))
    display(swing_tbl)

prods = DEEP_DIVE_PRODUCTS if DEEP_DIVE_PRODUCTS else sorted(co["product"].unique(), key=lambda p: (p != "All items (CPI-U)", p))

for prod in prods:
    display(Markdown(f"### {prod}"))
    a = ann_yoy[ann_yoy["product"] == prod].sort_values("periodYear")
    y = a.dropna(subset=["yoy_pct"])

    xs_cpi = a["periodYear"].astype(int).tolist()
    ys_cpi = pd.to_numeric(a["cpi"], errors="coerce").astype(float).tolist()
    fig_cpi = go.Figure(
        go.Scatter(
            x=xs_cpi,
            y=ys_cpi,
            mode="lines+markers",
            line=dict(color=PRISM[0], width=2),
            marker=dict(size=6),
        )
    )
    fig_cpi.update_layout(
        template="plotly_white",
        height=300,
        title=f"{prod} — annual CPI index (1982-84=100)",
        xaxis_title="Year",
        yaxis_title="CPI",
        showlegend=False,
    )
    show_plotly(fig_cpi)

    xs_y = y["periodYear"].astype(int).tolist()
    ys_y = y["yoy_pct"].astype(float).tolist()
    fig_yoy_bar = go.Figure(
        go.Bar(
            x=xs_y,
            y=ys_y,
            marker_color=[PRISM[2] if v >= 0 else PRISM[0] for v in ys_y],
        )
    )
    fig_yoy_bar.update_layout(
        template="plotly_white",
        height=300,
        title=f"{prod} — calendar-year YoY %",
        xaxis_title="Year",
        yaxis_title="YoY %",
        showlegend=False,
    )
    show_plotly(fig_yoy_bar)

    nb = max(4, min(14, len(y)))
    fig_hist = go.Figure(go.Histogram(x=y["yoy_pct"].astype(float).tolist(), nbinsx=nb))
    fig_hist.update_layout(
        template="plotly_white",
        height=280,
        title=f"{prod} — histogram of annual YoY % (n={len(y)})",
        xaxis_title="YoY %",
        yaxis_title="Count",
    )
    show_plotly(fig_hist)

    s = semi_all[semi_all["product"] == prod]
    fig_semi = go.Figure(
        go.Scatter(
            x=s["period_label"].astype(str).tolist(),
            y=pd.to_numeric(s["cpi"], errors="coerce").astype(float).tolist(),
            mode="lines+markers",
            line=dict(color=PRISM[4]),
            marker=dict(size=6),
        )
    )
    fig_semi.update_layout(
        template="plotly_white",
        height=320,
        title=f"{prod} — semi-annual CPI (H1 vs H2 path)",
        xaxis_tickangle=-50,
    )
    show_plotly(fig_semi)

    if "H1" in pv_spread.columns and "H2" in pv_spread.columns and "intra_year_swing_pct" in pv_spread.columns:
        try:
            sub_sw = pv_spread.loc[prod]
            fig_sw = go.Figure(
                go.Bar(
                    x=sub_sw.index.astype(str).tolist(),
                    y=pd.to_numeric(sub_sw["intra_year_swing_pct"], errors="coerce").astype(float).tolist(),
                    marker_color=PRISM[8],
                )
            )
            fig_sw.update_layout(
                title=f"{prod} — intra-year swing % (|H2−H1|/H1 by year)",
                template="plotly_white",
                height=300,
                xaxis_tickangle=-45,
                yaxis_title="Swing %",
            )
            show_plotly(fig_sw)
        except KeyError:
            pass

**Summary: intra-year swing** — mean / std / max of |H2−H1|/H1 (semi-annual, within calendar year)

,mean_swing_pct,std_swing_pct,max_swing_pct
product,,,
Fuels and utilities,4.32,3.32,10.53
Medical care,2.89,1.81,5.57
Apparel,2.84,4.34,11.40
Transportation,2.81,3.49,7.83
Recreation,1.88,2.13,6.21
Other goods and services,1.50,0.94,2.99
Education and communication,1.26,0.74,2.60
All items (CPI-U),1.25,0.50,2.16
Housing,1.24,1.53,4.50


### All items (CPI-U)

### Apparel

### Education and communication

...

### Housing

### Medical care

### Other goods and services

### Recreation

### Transportation